In [1]:
%pip install openai pdf2image pillow
%pip install dotenv


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from pdf2image import convert_from_path
import base64
import os
import re

In [20]:
load_dotenv()  # read .env file
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
filename = "images/jee_mains_page_5.jpg"
with open(filename, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")
    response = client.chat.completions.create (
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a math OCR assistant that converts scanned questions into Markdown with LaTeX."},
            {"role": "user", "content": [
                {"type": "text", "text": """
                                            Extract the math questions. Use Markdown and LaTeX for formulas.
                                            Mark the start of every question with a token '<start>' and end of every question with '<end>'.
                                            Extract all questions. Do not skip any, even if they look incomplete, unclear.
                                            If something resembles a question, include it.
                                            Some of the questions have multiple choices numbered (a), (b), (c) and so on.
                                            Include those as well, wherever they are visible.
                                            Your output must have the same number of <start>…<end> blocks as the number of questions visible in the input.
                                          """},
                {"type": "image_url", "image_url": 
                     {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ]
    )

In [21]:
print(response.choices[0].message.content)

```markdown
<start> If \( f(x) + 4\left( \frac{1}{x} \right) = x^2 - 2 - \sqrt{x} \) and \( y = 9x^2 f(x) \), then \( y \) is strictly increasing in [Single Correct Type, 2024 Main] 
  (a) \( (0, \frac{1}{\sqrt{5}}, \infty) \) 
  (b) \( (1, \frac{1}{\sqrt{5}}, 0) \) 
  (c) \( (-\frac{1}{\sqrt{5}}, 0) \) 
  (d) \( (-\infty, -\frac{1}{\sqrt{5}}, 0) \) <end>

<start> Let the sum of the maximum and the minimum values of the function \( f(x) = \frac{2x^2 - 3x + 8}{2x^2 + 3x + 8} \) be \( \frac{m}{n} \), where gcd(m, n) = 1. 
Then, \( m + n \) is equal to [Single Correct Type, 2024 Main] 
  (a) 20 
  (b) 217 
  (c) 195 
  (d) 182 <end>

<start> Let \( f(x) = (x + 3)^2 - (x - 2)^3, \, x \in [-4, 4] \). If \( M \) and \( m \) are the maximum and minimum values of \( f \), respectively in \( [-4, 4] \), then the value of \( M - m \) is [Single Correct Type, 2024 Main] 
  (a) 392 
  (b) 108 
  (c) 600 <end>

<start> The function \( f(x) = 2x + 3x^{2/3}, \, x \in R \) has [Single Correct Type, 20

In [22]:
text = response.choices[0].message.content
pattern = r'<start>(.*?)<end>'
matches = re.findall(pattern, text, flags=re.DOTALL)

with open('output/jee_mains_page_5.md', 'w') as of:
    for match in matches:
        question = ' '.join(match.splitlines())
        of.writelines(question)
        of.write('\n')